## Import Libraries

In [ ]:
import os
import json
from google.colab import drive
import json
import pandas as pd
from itertools import product
from collections import defaultdict

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Read Processed JSON Files

Read all JSON Files, and Combined them into master JSON for easier access.

In [ ]:
# 1. Define your folder path
folder_path = '/content/drive/MyDrive/BT4222 Project/FinalAudited'

# 2. Create an empty list to hold all your combined data
combined_audit_data = []

# 3. Loop through all files in the directory
for filename in os.listdir(folder_path):
    # Only process JSON files
    if filename.endswith('.json'):
        file_path = os.path.join(folder_path, filename)

        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                data = json.load(file)

                # If the file contains a list of records (like your sample), extend the master list
                if isinstance(data, list):
                    combined_audit_data.extend(data)
                # If the file contains a single dictionary record, append it to the master list
                else:
                    combined_audit_data.append(data)

        except json.JSONDecodeError:
            print(f"Warning: Could not decode JSON in file {filename}.")
        except Exception as e:
            print(f"Error reading {filename}: {e}")

# 4. Check how many total records we collected
print(f"Successfully combined {len(combined_audit_data)} total records!")

# 5. Save the combined data into a single master JSON file for easy access later
output_path = '/content/drive/MyDrive/BT4222 Project/Data/Master_Audit_Dataset.json'

with open(output_path, 'w', encoding='utf-8') as outfile:
    json.dump(combined_audit_data, outfile, indent=2)

print(f"Master file saved to: {output_path}")

Successfully combined 484 total records!
Master file saved to: /content/drive/MyDrive/BT4222 Project/Data/Master_Audit_Dataset.json


# Transform JSON to CSV
For every case number, there is a row for each **unique** plaintiff and defendant pair. Each row will contain the metadata, facts, IRAC Framework and conclusion for both pairs. To make the distinction between rows more for each case, the facts are combined, and the IRAC framework are chosen from the plaintiff side.

Note: Defendant Label is labled as Liable or Not Liable given at a case level.

In [ ]:
# 1. Load the master dataset we created in the previous step
file_path = '/content/drive/MyDrive/BT4222 Project/Data/Master_Audit_Dataset.json'

with open(file_path, 'r', encoding='utf-8') as f:
    master_data = json.load(f)

# 2. Group all records by Case_Number
cases_grouped = defaultdict(list)
for record in master_data:
    case_num = record.get("Metadata", {}).get("Case_Number")
    if case_num:
        cases_grouped[case_num].append(record)

rows = []

# 3. Process each case group individually
for case_number, records in cases_grouped.items():

    plaintiffs = [r for r in records if r.get("Party_Details", {}).get("Role") == "Plaintiff"]
    defendants = [r for r in records if r.get("Party_Details", {}).get("Role") == "Defendant"]

    for p, d in product(plaintiffs, defendants):
        meta = p.get('Metadata', {})
        p_details = p.get('Party_Details', {})
        d_details = d.get('Party_Details', {})

        row = {
            'Case_Number': meta.get('Case_Number'),
            'Coram': meta.get('Coram'),
            'Judge': meta.get('Judge'),
            'Date': meta.get('Date'),
            'Tribunal_Court': meta.get('Tribunal_Court'),
            'Plaintiff_Name': p_details.get('Name'),
            'Defendant_Name': d_details.get('Name'),
            'Combined_Facts': [p_details.get('Facts', []), d_details.get('Facts', [])],
            'Combined_Issue': [p_details.get('Issue', ''), d_details.get('Issue', '')],
            'Combined_Rule': [p_details.get('Rule', ''), d_details.get('Rule', '')],
            'Combined_Application': [p_details.get('Application', ''), d_details.get('Application', '')],
            'plaintiff_label': p.get('Label'),
            'defendant_label': d.get('Label')
        }

        rows.append(row)

# 4. Create the final DataFrame
df = pd.DataFrame(rows)

# Convert 'Date' to datetime objects for accurate sorting (ignores blank/bad formats)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Sort by Date (ascending order: oldest to newest).
# Note: To sort newest to oldest, use: df.sort_values(by='Date', ascending=False)
df = df.sort_values(by='Date')

# Convert the Date back to a clean string format (YYYY-MM-DD) for the CSV
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')

# View the total number of pairings and the first few rows
print(f"Total unique Plaintiff-Defendant pairs processed: {len(df)}")
display(df.head())

Total unique Plaintiff-Defendant pairs processed: 358


,Case_Number,Coram,Judge,Date,Tribunal_Court,Plaintiff_Name,Defendant_Name,Combined_Facts,Combined_Issue,Combined_Rule,Combined_Application,plaintiff_label,defendant_label
263,CA 206/1999 (Counterclaim),Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-03,Court of Appeal,Gan Boon Hock,Kea Resources Pte Ltd,"[[{'Fact_Type': 'CONTRACT_EVENT', 'Fact_Date':...",[Whether an employee is entitled to claim unpa...,[An employee may claim financial entitlements ...,[He was owed financial entitlements as per his...,Claim Allowed,Liable
262,CA 206/1999,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-03,Court of Appeal,Kea Holdings Pte Ltd,Gan Boon Hock,"[[{'Fact_Type': 'CORPORATE_ROLE', 'Fact_Date':...",[Whether a director can be held legally respon...,[A director must act in the best interests of ...,[Gan diverted business and misrepresented sale...,Claim Allowed in Part,Liable
353,Civil Appeal No 124 of 1999,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-28,Court of Appeal,Kumagai Property Marketing Pte Ltd,Low Hua Kin,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...",[Whether a subsidiary company can hold its dir...,[A director is legally responsible for losses ...,[The Plaintiff stated that it was used as a me...,Claim Allowed,Liable
352,Civil Appeal No 124 of 1999,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-28,Court of Appeal,Kumagai-Zenecon Construction Pte Ltd (in liqui...,Low Hua Kin,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...",[Whether a joint venture company can recover l...,[A director’s duty to the parent company exten...,[The Plaintiff stated that the director’s unau...,Claim Allowed,Liable
58,Suit 1032/1999,Kan Ting Chiu J,Kan Ting Chiu,2000-08-26,High Court,Jurong Readymix Concrete Pte Ltd,Chng Heng Tiu,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...",[Whether a corporate guarantor is liable for u...,[A guarantee is enforceable if supported by va...,[A concrete supplier increased a contractor's ...,Claim Allowed,Liable


In [ ]:
# 1. Extract Plaintiff's Issue, Rule, and Application
# Since your script saved them as [plaintiff_string, defendant_string], we just grab index 0.
df['Issue'] = df['Combined_Issue'].str[0]
df['Rule'] = df['Combined_Rule'].str[0]
df['Application'] = df['Combined_Application'].str[0]

# Drop the old array-based columns to keep the dataframe clean
df = df.drop(columns=['Combined_Issue', 'Combined_Rule', 'Combined_Application'])

# 2. Merge and sort the nested facts arrays
def merge_and_sort_facts(facts_pair):
    # Ensure it's a list containing the [plaintiff_facts, defendant_facts] arrays
    if isinstance(facts_pair, list) and len(facts_pair) == 2:
        # Combine the two lists into one flat list
        combined = facts_pair[0] + facts_pair[1]

        # Sort chronologically by Fact_Date. Missing dates are safely pushed to the end.
        return sorted(combined, key=lambda x: x.get('Fact_Date', '9999-99-99'))
    return []

# Apply the function to the facts column
df['Combined_Facts'] = df['Combined_Facts'].apply(merge_and_sort_facts)

# 3. Sort the overall DataFrame by the Case Date
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.sort_values(by='Date')
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')

# Preview the cleaned data
display(df.head())

,Case_Number,Coram,Judge,Date,Tribunal_Court,Plaintiff_Name,Defendant_Name,Combined_Facts,plaintiff_label,defendant_label,Issue,Rule,Application
263,CA 206/1999 (Counterclaim),Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-03,Court of Appeal,Gan Boon Hock,Kea Resources Pte Ltd,"[{'Fact_Type': 'CONTRACT_EVENT', 'Fact_Date': ...",Claim Allowed,Liable,Whether an employee is entitled to claim unpai...,An employee may claim financial entitlements i...,He was owed financial entitlements as per his ...
262,CA 206/1999,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-03,Court of Appeal,Kea Holdings Pte Ltd,Gan Boon Hock,"[{'Fact_Type': 'CORPORATE_ROLE', 'Fact_Date': ...",Claim Allowed in Part,Liable,Whether a director can be held legally respons...,A director must act in the best interests of t...,Gan diverted business and misrepresented sale ...
353,Civil Appeal No 124 of 1999,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-28,Court of Appeal,Kumagai Property Marketing Pte Ltd,Low Hua Kin,"[{'Fact_Type': 'CORPORATE_ROLE', 'Fact_Date': ...",Claim Allowed,Liable,Whether a subsidiary company can hold its dire...,A director is legally responsible for losses i...,The Plaintiff stated that it was used as a mer...
352,Civil Appeal No 124 of 1999,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-28,Court of Appeal,Kumagai-Zenecon Construction Pte Ltd (in liqui...,Low Hua Kin,"[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '198...",Claim Allowed,Liable,Whether a joint venture company can recover lo...,A director’s duty to the parent company extend...,The Plaintiff stated that the director’s unaut...
58,Suit 1032/1999,Kan Ting Chiu J,Kan Ting Chiu,2000-08-26,High Court,Jurong Readymix Concrete Pte Ltd,Chng Heng Tiu,"[{'Fact_Type': 'CORPORATE_ROLE', 'Fact_Date': ...",Claim Allowed,Liable,Whether a corporate guarantor is liable for un...,A guarantee is enforceable if supported by val...,A concrete supplier increased a contractor's c...


In [ ]:
court_mapping = {
    "High Court": "High Court of Singapore",
    "High Court of the Republic of Singapore": "High Court of Singapore",
    "High Court of Singapore": "High Court of Singapore",

    "General Division of the High Court of the Republic of Singapore": "General Division of the High Court of Singapore",
    "General Division of the High Court of Singapore": "General Division of the High Court of Singapore",

    "Court of Appeal": "Court of Appeal",
    "Court of Appeal of the Republic of Singapore": "Court of Appeal",

    "Singapore International Commercial Court": "Singapore International Commercial Court",

    "Appellate Division of the High Court of Singapore": "Appellate Division of the High Court of Singapore",
}

df["Tribunal_Court"] = df["Tribunal_Court"].map(court_mapping).fillna(df["Tribunal_Court"])


## Label Checking
Initially we wanted to predict the case as whole so the defendant is labled liable as long (if multiple plaintiffs) he is liable for minimally one plaintiff. But this limits information. We extracted the labels are a pair and map it to check

In [ ]:
import json
import pandas as pd
from pathlib import Path  # <-- Make sure this is imported!

# 1. Define the path to your Label_Checks folder using Path()
folder_path = Path("/content/drive/MyDrive/BT4222 Project/Label_Checks")

all_records = []

# 2. Iterate through every .json file in the directory
for filepath in folder_path.glob("*.json"):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)

            # If the JSON file contains a list of records, extend our master list
            if isinstance(data, list):
                all_records.extend(data)
            # If the JSON file is just a single dictionary, append it
            else:
                all_records.append(data)

    except json.JSONDecodeError:
        print(f"Skipping {filepath.name}: Invalid JSON format.")
    except Exception as e:
        print(f"Error reading {filepath.name}: {e}")

# 3. Convert the combined data into a pandas DataFrame
labels = pd.DataFrame(all_records)

In [ ]:
# 1. Prepare the labels DataFrame for joining
# We rename the keys to match 'df' and append '_check' to the target labels
labels_prepared = labels.rename(columns={
    'Plaintiff': 'Plaintiff_Name',
    'Defendant': 'Defendant_Name',
    'Plaintiff_Label': 'Plaintiff_Label_check',
    'Defendant_Label': 'Defendant_Label_check'
})

# 2. Perform the left join
# This keeps all rows in 'df' and adds the check columns where a match is found
final_merged_df = pd.merge(
    df,
    labels_prepared[[
        'Case_Number',
        'Plaintiff_Name',
        'Defendant_Name',
        'Plaintiff_Label_check',
        'Defendant_Label_check'
    ]],
    on=['Case_Number', 'Plaintiff_Name', 'Defendant_Name'],
    how='left'
)

# 3. Validation
print(f"Join complete. Total rows: {len(final_merged_df)}")
print(f"Rows with new label checks: {final_merged_df['Plaintiff_Label_check'].notna().sum()}")

# View a sample of the results
display(final_merged_df.head())

Join complete. Total rows: 361
Rows with new label checks: 358


,Case_Number,Coram,Judge,Date,Tribunal_Court,Plaintiff_Name,Defendant_Name,Combined_Facts,plaintiff_label,defendant_label,Issue,Rule,Application,Plaintiff_Label_check,Defendant_Label_check
0,CA 206/1999 (Counterclaim),Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-03,Court of Appeal,Gan Boon Hock,Kea Resources Pte Ltd,"[{'Fact_Type': 'CONTRACT_EVENT', 'Fact_Date': ...",Claim Allowed,Liable,Whether an employee is entitled to claim unpai...,An employee may claim financial entitlements i...,He was owed financial entitlements as per his ...,Claim Allowed,Liable
1,CA 206/1999,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-03,Court of Appeal,Kea Holdings Pte Ltd,Gan Boon Hock,"[{'Fact_Type': 'CORPORATE_ROLE', 'Fact_Date': ...",Claim Allowed in Part,Liable,Whether a director can be held legally respons...,A director must act in the best interests of t...,Gan diverted business and misrepresented sale ...,Claim Allowed In-part,Liable
2,Civil Appeal No 124 of 1999,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-28,Court of Appeal,Kumagai Property Marketing Pte Ltd,Low Hua Kin,"[{'Fact_Type': 'CORPORATE_ROLE', 'Fact_Date': ...",Claim Allowed,Liable,Whether a subsidiary company can hold its dire...,A director is legally responsible for losses i...,The Plaintiff stated that it was used as a mer...,Claim Allowed,Liable
3,Civil Appeal No 124 of 1999,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-28,Court of Appeal,Kumagai-Zenecon Construction Pte Ltd (in liqui...,Low Hua Kin,"[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '198...",Claim Allowed,Liable,Whether a joint venture company can recover lo...,A director’s duty to the parent company extend...,The Plaintiff stated that the director’s unaut...,Claim Allowed,Liable
4,Suit 1032/1999,Kan Ting Chiu J,Kan Ting Chiu,2000-08-26,High Court of Singapore,Jurong Readymix Concrete Pte Ltd,Chng Heng Tiu,"[{'Fact_Type': 'CORPORATE_ROLE', 'Fact_Date': ...",Claim Allowed,Liable,Whether a corporate guarantor is liable for un...,A guarantee is enforceable if supported by val...,A concrete supplier increased a contractor's c...,Claim Allowed,Liable


In [ ]:
# 1. STANDARDIZE THE LABELS DATAFRAME FIRST


# We consolidate 'Conclusion'/'Label' into 'Plaintiff_Label'/'Defendant_Label'
# so we don't end up with duplicate column names.
labels_clean = labels.copy()

if 'Conclusion' in labels_clean.columns:
    labels_clean['Plaintiff_Label'] = labels_clean['Plaintiff_Label'].fillna(labels_clean['Conclusion'])
if 'Label' in labels_clean.columns:
    labels_clean['Defendant_Label'] = labels_clean['Defendant_Label'].fillna(labels_clean['Label'])

# Now rename for the join
labels_prepared = labels_clean.rename(columns={
    'Plaintiff': 'Plaintiff_Name',
    'Defendant': 'Defendant_Name',
    'Plaintiff_Label': 'Plaintiff_Label_check',
    'Defendant_Label': 'Defendant_Label_check'
})

# Keep ONLY the columns we need (removes the old 'Conclusion'/'Label' columns)
labels_prepared = labels_prepared[[
    'Case_Number', 'Plaintiff_Name', 'Defendant_Name',
    'Plaintiff_Label_check', 'Defendant_Label_check'
]]

# Drop duplicates to prevent row expansion
labels_prepared = labels_prepared.drop_duplicates(subset=['Case_Number', 'Plaintiff_Name', 'Defendant_Name'])

# 2. PERFORM THE JOIN

# Standardize whitespace in join keys for both dfs
for col in ['Case_Number', 'Plaintiff_Name', 'Defendant_Name']:
    df[col] = df[col].astype(str).str.strip()
    labels_prepared[col] = labels_prepared[col].astype(str).str.strip()

final_merged_df = pd.merge(
    df,
    labels_prepared,
    on=['Case_Number', 'Plaintiff_Name', 'Defendant_Name'],
    how='left'
)

# 3. VALIDATION (SCALAR LOGIC)

total_rows = len(final_merged_df)
# .sum().max() ensures we get a single number even if something goes wrong
matched_count = int(final_merged_df['Plaintiff_Label_check'].notna().sum())

print(f"Successfully joined! Merged dataset has {total_rows} rows.")
print(f"Rows with a label match: {matched_count}")

if (total_rows - matched_count) > 0:
    print(f"Rows missing checks: {total_rows - matched_count}")
    unmatched = final_merged_df[final_merged_df['Plaintiff_Label_check'].isna()]
    display(unmatched[['Case_Number', 'Plaintiff_Name', 'Defendant_Name']].head(10))
else:
    print("Perfect match! All rows have labels.")

Successfully joined! Merged dataset has 358 rows.
Rows with a label match: 358
Perfect match! All rows have labels.


In [ ]:
final_merged_df

,Case_Number,Coram,Judge,Date,Tribunal_Court,Plaintiff_Name,Defendant_Name,Combined_Facts,plaintiff_label,defendant_label,Issue,Rule,Application,Plaintiff_Label_check,Defendant_Label_check
0,CA 206/1999 (Counterclaim),Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-03,Court of Appeal,Gan Boon Hock,Kea Resources Pte Ltd,"[{'Fact_Type': 'CONTRACT_EVENT', 'Fact_Date': ...",Claim Allowed,Liable,Whether an employee is entitled to claim unpai...,An employee may claim financial entitlements i...,He was owed financial entitlements as per his ...,Claim Allowed,Liable
1,CA 206/1999,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-03,Court of Appeal,Kea Holdings Pte Ltd,Gan Boon Hock,"[{'Fact_Type': 'CORPORATE_ROLE', 'Fact_Date': ...",Claim Allowed in Part,Liable,Whether a director can be held legally respons...,A director must act in the best interests of t...,Gan diverted business and misrepresented sale ...,Claim Allowed In-part,Liable
2,Civil Appeal No 124 of 1999,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-28,Court of Appeal,Kumagai Property Marketing Pte Ltd,Low Hua Kin,"[{'Fact_Type': 'CORPORATE_ROLE', 'Fact_Date': ...",Claim Allowed,Liable,Whether a subsidiary company can hold its dire...,A director is legally responsible for losses i...,The Plaintiff stated that it was used as a mer...,Claim Allowed,Liable
3,Civil Appeal No 124 of 1999,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-28,Court of Appeal,Kumagai-Zenecon Construction Pte Ltd (in liqui...,Low Hua Kin,"[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '198...",Claim Allowed,Liable,Whether a joint venture company can recover lo...,A director’s duty to the parent company extend...,The Plaintiff stated that the director’s unaut...,Claim Allowed,Liable
4,Suit 1032/1999,Kan Ting Chiu J,Kan Ting Chiu,2000-08-26,High Court of Singapore,Jurong Readymix Concrete Pte Ltd,Chng Heng Tiu,"[{'Fact_Type': 'CORPORATE_ROLE', 'Fact_Date': ...",Claim Allowed,Liable,Whether a corporate guarantor is liable for un...,A guarantee is enforceable if supported by val...,A concrete supplier increased a contractor's c...,Claim Allowed,Liable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
353,Suit No 942 of 2021,Mohamed Faizal JC,Mohamed Faizal,2025-07-29,General Division of the High Court of Singapore,Toh Ai Ling,Lee Si Ye,"[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '201...",Claim Allowed,Not Liable,Whether a joint liquidator shares the collecti...,Joint and several liquidators share authority ...,"As part of the liquidator team, Toh Ai Ling pa...",Claim Allowed In-part,Liable
354,Suit No 942 of 2021,Mohamed Faizal JC,Mohamed Faizal,2025-07-29,General Division of the High Court of Singapore,Toh Ai Ling,Ju Xiao,"[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '201...",Claim Allowed,Liable,Whether a joint liquidator shares the collecti...,Joint and several liquidators share authority ...,"As part of the liquidator team, Toh Ai Ling pa...",Claim Allowed In-part,Liable
355,Suit No 942 of 2021,Mohamed Faizal JC,Mohamed Faizal,2025-07-29,General Division of the High Court of Singapore,Toh Ai Ling,Cheong Ming Feng,"[{'Fact_Type': 'FINANCIAL_EVENT', 'Fact_Date':...",Claim Allowed,Not Liable,Whether a joint liquidator shares the collecti...,Joint and several liquidators share authority ...,"As part of the liquidator team, Toh Ai Ling pa...",Claim Allowed In-part,Liable
356,Suit No 942 of 2021,Mohamed Faizal JC,Mohamed Faizal,2025-07-29,General Division of the High Court of Singapore,Bob Yap Cheng Ghee,Lee Si Ye,"[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '201...",Claim Allowed,Not Liable,Whether a court-appointed liquidator has the a...,Liquidators are empowered under the Insolvency...,As a joint liquidator appointed on 16 August 2...,Claim Allowed In-part,Liable


In [ ]:
# 1. Drop the original labels (if they exist)
final_merged_df = final_merged_df.drop(columns=['plaintiff_label', 'defendant_label'], errors='ignore')

# 2. Rename the check columns to the standard label names
final_merged_df = final_merged_df.rename(columns={
    'Plaintiff_Label_check': 'Plaintiff_Label',
    'Defendant_Label_check': 'Defendant_Label'
})

# 3. Check the results
print(f"Updated columns: {final_merged_df.columns.tolist()}")
final_merged_df.head()

Updated columns: ['Case_Number', 'Coram', 'Judge', 'Date', 'Tribunal_Court', 'Plaintiff_Name', 'Defendant_Name', 'Combined_Facts', 'Issue', 'Rule', 'Application', 'Plaintiff_Label', 'Defendant_Label']


,Case_Number,Coram,Judge,Date,Tribunal_Court,Plaintiff_Name,Defendant_Name,Combined_Facts,Issue,Rule,Application,Plaintiff_Label,Defendant_Label
0,CA 206/1999 (Counterclaim),Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-03,Court of Appeal,Gan Boon Hock,Kea Resources Pte Ltd,"[{'Fact_Type': 'CONTRACT_EVENT', 'Fact_Date': ...",Whether an employee is entitled to claim unpai...,An employee may claim financial entitlements i...,He was owed financial entitlements as per his ...,Claim Allowed,Liable
1,CA 206/1999,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-03,Court of Appeal,Kea Holdings Pte Ltd,Gan Boon Hock,"[{'Fact_Type': 'CORPORATE_ROLE', 'Fact_Date': ...",Whether a director can be held legally respons...,A director must act in the best interests of t...,Gan diverted business and misrepresented sale ...,Claim Allowed In-part,Liable
2,Civil Appeal No 124 of 1999,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-28,Court of Appeal,Kumagai Property Marketing Pte Ltd,Low Hua Kin,"[{'Fact_Type': 'CORPORATE_ROLE', 'Fact_Date': ...",Whether a subsidiary company can hold its dire...,A director is legally responsible for losses i...,The Plaintiff stated that it was used as a mer...,Claim Allowed,Liable
3,Civil Appeal No 124 of 1999,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,Chao Hick Tin JA; L P Thean JA; Yong Pung How CJ,2000-07-28,Court of Appeal,Kumagai-Zenecon Construction Pte Ltd (in liqui...,Low Hua Kin,"[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '198...",Whether a joint venture company can recover lo...,A director’s duty to the parent company extend...,The Plaintiff stated that the director’s unaut...,Claim Allowed,Liable
4,Suit 1032/1999,Kan Ting Chiu J,Kan Ting Chiu,2000-08-26,High Court of Singapore,Jurong Readymix Concrete Pte Ltd,Chng Heng Tiu,"[{'Fact_Type': 'CORPORATE_ROLE', 'Fact_Date': ...",Whether a corporate guarantor is liable for un...,A guarantee is enforceable if supported by val...,A concrete supplier increased a contractor's c...,Claim Allowed,Liable


In [ ]:
#Export to CSV
output_csv = '/content/drive/MyDrive/BT4222 Project/Data/Court_Cases.csv'
final_merged_df.to_csv(output_csv, index=False)